# Training & Evaluasi YOLOv8-Seg: Panel dan Kerusakan Mobil

Dataset: Kaggle *Car parts & car damages* (format Supervisely) di `data/raw/`, ditambah **CarDD** (Wang et al., 2023; Hugging Face `harpreetsahota/CarDD`, 2.816 foto) di `data/raw/CarDD_hf/` khusus untuk model kerusakan.

| Model | Folder asli (**nama tertukar**) | Kelas | Output |
|---|---|---|---|
| Panel | `Car damages dataset` | 21 (Front-door, Hood, Fender, ...) | `weights/parts_best.pt` |
| Kerusakan | `Car parts dataset` + CarDD | 2 (Dent, Scratch); kelas kerusakan lain diabaikan | `weights/damage_best.pt` |

441 foto punya label panel **dan** kerusakan; 66 di antaranya ada di split test dan dipakai untuk evaluasi end-to-end (bagian 4).

> Training di CPU laptop kira-kira 10+ jam per model. Disarankan pakai **Google Colab (GPU T4)**: upload folder project (termasuk `data/raw/`) ke Google Drive, lalu jalankan sel Colab di bawah.

In [ ]:
# (Colab) mount Drive dan pindah ke folder project
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/skripsi-body-repair
# !pip install -q ultralytics transformers

import os, shutil
import torch
from ultralytics import YOLO

print('CUDA tersedia:', torch.cuda.is_available())
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
EPOCHS = 100     # kecilkan (mis. 3) untuk uji coba cepat
IMGSZ = 640
BATCH = 16 if torch.cuda.is_available() else 4
os.makedirs('weights', exist_ok=True)

## 1. Konversi dataset Supervisely → YOLO-seg
Hasil: `data/annotated/parts/`, `data/annotated/damage/`, dan `data/annotated/split_info.json`
(split 70/15/15, seed 42; foto bersama selalu berada di split yang sama untuk kedua model).

In [ ]:
from src.convert_dataset import convert

summary = convert(seed=42)
summary

## 2. Training model panel

In [ ]:
def train(data_yaml, name):
    model = YOLO('yolov8s-seg.pt')
    results = model.train(
        data=data_yaml, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE,
        patience=20, project='runs', name=name, exist_ok=True,
    )
    shutil.copy(os.path.join(results.save_dir, 'weights', 'best.pt'), f'weights/{name}_best.pt')
    print(f'Tersimpan: weights/{name}_best.pt')

train('data/annotated/parts/data.yaml', 'parts')

## 3. Training model kerusakan
Hanya kelas **Dent** dan **Scratch** (lihat `CLASS_FILTER` di `src/convert_dataset.py`); anotasi kelas kerusakan lain dianggap latar belakang.

In [ ]:
train('data/annotated/damage/data.yaml', 'damage')

## 4. Evaluasi
### 4a. Segmentasi per model (split test)
Metrik utama: **mask mAP50** dan **mask mAP50-95**, juga per kelas.

In [ ]:
import pandas as pd

def evaluate_seg(weights, data_yaml):
    m = YOLO(weights).val(data=data_yaml, split='test', device=DEVICE, plots=False, verbose=False)
    overall = {
        'box mAP50': m.box.map50, 'box mAP50-95': m.box.map,
        'mask mAP50': m.seg.map50, 'mask mAP50-95': m.seg.map,
    }
    per_class = pd.DataFrame({
        'kelas': [m.names[i] for i in m.ap_class_index],
        'mask mAP50': m.seg.ap50,
        'mask mAP50-95': m.seg.ap,
    }).round(4)
    return overall, per_class

evals = [
    ('parts', 'data/annotated/parts/data.yaml'),
    ('damage', 'data/annotated/damage/data.yaml'),          # gabungan
    ('damage', 'data/annotated/damage/data_kaggle.yaml'),   # hanya foto Kaggle
    ('damage', 'data/annotated/damage/data_cardd.yaml'),    # hanya foto CarDD
]
for name, data_yaml in evals:
    overall, per_class = evaluate_seg(f'weights/{name}_best.pt', data_yaml)
    print(f'=== {name} | {data_yaml} ===', {k: round(v, 4) for k, v in overall.items()})
    display(per_class)

### 4b. Evaluasi end-to-end (66 foto test berlabel panel + kerusakan)
Biaya dari **anotasi manual** (ground truth) dibandingkan dengan biaya dari **prediksi model**,
memakai formula dan depth map yang sama.
- *panel precision/recall*: apakah panel yang rusak teridentifikasi dengan benar
- *A_mask MAE*: selisih rasio luas kerusakan pada panel yang cocok
- *cost MAE / MAPE*: selisih estimasi biaya (Rp dan %)

In [ ]:
from src.pipeline import BodyRepairPipeline
from src.evaluate import evaluate_end_to_end

pipe = BodyRepairPipeline()
per_panel, e2e_summary = evaluate_end_to_end(pipe, split='test')
display(pd.Series(e2e_summary, name='nilai').to_frame())
per_panel.head(20)

## 5. Contoh hasil pipeline

In [ ]:
import cv2, glob, matplotlib.pyplot as plt

test_imgs = sorted(glob.glob('data/annotated/damage/images/test/*'))[:4]
for path in test_imgs:
    res = pipe.analyze(cv2.imread(path))
    plt.figure(figsize=(6, 4))
    plt.imshow(cv2.cvtColor(res['overlay'], cv2.COLOR_BGR2RGB)); plt.title(os.path.basename(path)); plt.axis('off')
    plt.show()
    for it in res['items']:
        print(it['panel_key'] or it['part_class'], it['damage_classes'],
              'A=%.3f D=%.3f' % (it['mask_area_ratio'], it['depth_deviation']),
              it['cost']['formatted_total'] if it['cost'] else it['status'])
    print('TOTAL', res['formatted_total'], '| kerusakan tanpa panel:', len(res['unassigned']))